# ESP32-S3 Flash Manager

Manages firmware and SPIFFS flashing for the **BugBuster ESP32-S3** via PlatformIO.  
Ports are auto-detected; manual override is supported.

| Cell | Action |
|------|--------|
| Port detection | List all serial ports, auto-select ESP32-S3 |
| Flash firmware | `pio run -t upload` — full firmware |
| Flash SPIFFS | `pio run -t uploadfs` — SPIFFS partition only |
| Erase flash | Full chip erase |
| Serial monitor | Live serial output at 115200 baud |

**Partition layout (partitions.csv)**

| Name | Type | Offset | Size |
|------|------|--------|------|
| nvs | data/nvs | 0x9000 | 20 KB |
| otadata | data/ota | 0xE000 | 8 KB |
| app0 | app/ota_0 | 0x10000 | 2.5 MB |
| spiffs | data/spiffs | 0x290000 | 1.4 MB |

In [ ]:
import subprocess
import os
import sys
import time
import threading
from pathlib import Path

try:
    import serial
    import serial.tools.list_ports
    SERIAL_AVAILABLE = True
except ImportError:
    SERIAL_AVAILABLE = False
    print("pyserial not found — install with: pip install pyserial")

# Auto-detect firmware directory relative to this notebook.
# Works on any OS: notebook is at <repo>/notebooks/, firmware at <repo>/Firmware/esp32_ad74416h/
_notebook_dir = Path.cwd()
_candidates = [
    _notebook_dir / ".." / "Firmware" / "esp32_ad74416h",          # run from notebooks/
    _notebook_dir / "Firmware" / "esp32_ad74416h",                  # run from repo root
    _notebook_dir.parent / "Firmware" / "esp32_ad74416h",           # notebook opened from subdir
]
# Platform-specific fallbacks
if sys.platform == "win32":
    _candidates.append(Path(r"C:\Users\Public\Documents\Altium\BugBuster\Firmware\esp32_ad74416h"))
elif sys.platform == "darwin":
    _candidates.append(Path.home() / "Desktop" / "BugBuster" / "Firmware" / "esp32_ad74416h")

FIRMWARE_DIR = None
for c in _candidates:
    if c is not None:
        resolved = c.resolve()
        if resolved.exists() and (resolved / "platformio.ini").exists():
            FIRMWARE_DIR = resolved
            break

if FIRMWARE_DIR is None:
    print("⚠ Could not find firmware directory automatically.")
    print("  Set it manually: FIRMWARE_DIR = Path('/path/to/Firmware/esp32_ad74416h')")
    FIRMWARE_DIR = Path(".")  # fallback so the rest of the notebook doesn't crash

PIO_ENV      = "esp32s3"
BAUD         = 115200

# Known USB VID:PID for ESP32 chips
ESP32_VIDS = {
    0x10C4: "Silicon Labs CP210x",
    0x1A86: "CH340/CH341",
    0x0403: "FTDI",
    0x303A: "Espressif (native USB)",
    0x2341: "Arduino",
}

print(f"Firmware directory : {FIRMWARE_DIR}")
print(f"Directory exists   : {FIRMWARE_DIR.exists()}")
print(f"PlatformIO env     : {PIO_ENV}")
print(f"pyserial available : {SERIAL_AVAILABLE}")

## Port Detection
Scans all serial ports and ranks them by likelihood of being an ESP32.  
`UPLOAD_PORT` and `MONITOR_PORT` are set automatically; override below if needed.

In [ ]:
def detect_esp32_ports():
    """Return (upload_port, monitor_port) by scanning serial ports.
    
    The BugBuster firmware exposes two CDC interfaces:
      - CLI / upload port  (first CDC)
      - UART bridge port   (second CDC)
    When using a UART-USB adapter the two ports may be the same physical chip.
    """
    if not SERIAL_AVAILABLE:
        return None, None

    all_ports = list(serial.tools.list_ports.comports())

    print(f"Found {len(all_ports)} serial port(s):")
    print(f"  {'Port':<10} {'VID:PID':<12} {'Description'}")
    print("  " + "-" * 60)
    for p in all_ports:
        vid_pid = f"{p.vid:04X}:{p.pid:04X}" if p.vid else "N/A"
        print(f"  {p.device:<10} {vid_pid:<12} {p.description}")

    # Score ports: higher = more likely to be ESP32
    def score(p):
        s = 0
        desc = (p.description or "").lower()
        mfr  = (p.manufacturer or "").lower()
        if p.vid in ESP32_VIDS:
            s += 10
        if p.vid == 0x303A:   # Espressif native USB — highest confidence
            s += 20
        for kw in ["esp32", "espressif", "cp210", "ch340", "ch341", "ftdi", "usb serial"]:
            if kw in desc or kw in mfr:
                s += 5
        return s

    ranked = sorted(all_ports, key=score, reverse=True)
    esp_ports = [p for p in ranked if score(p) > 0]

    if not esp_ports:
        print("\nNo ESP32-like port found. Connect the device and re-run.")
        return None, None

    # If two ESP32-like ports found (dual CDC), assign upload=first, monitor=second
    upload_port  = esp_ports[0].device
    monitor_port = esp_ports[1].device if len(esp_ports) >= 2 else esp_ports[0].device

    print(f"\nAuto-selected:")
    print(f"  Upload port  : {upload_port}")
    print(f"  Monitor port : {monitor_port}")
    return upload_port, monitor_port


UPLOAD_PORT, MONITOR_PORT = detect_esp32_ports()

# ── Manual override (uncomment and set if auto-detection is wrong) ────────────
# UPLOAD_PORT  = "COM10"
# MONITOR_PORT = "COM13"

In [ ]:
# ─── Shared streaming helper ──────────────────────────────────────────────────
_active_proc = None

def run_pio(extra_args, port=None, env=PIO_ENV):
    """Run a PlatformIO command with live output streaming."""
    global _active_proc

    if port is None:
        port = UPLOAD_PORT
    if port is None:
        print("ERROR: No port selected. Run port detection first or set UPLOAD_PORT manually.")
        return 1

    cmd = ["pio", "run", "-e", env] + extra_args + ["--upload-port", port]
    print(f"$ {' '.join(cmd)}")
    print("=" * 60)

    proc = subprocess.Popen(
        cmd,
        cwd=str(FIRMWARE_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ},
    )
    _active_proc = proc
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate()
        print("\n[Interrupted]")
    finally:
        proc.wait()
        _active_proc = None

    print("=" * 60)
    if proc.returncode == 0:
        print("Done.")
    else:
        print(f"FAILED (exit code {proc.returncode})")
    return proc.returncode

print("run_pio() helper ready.")

## Flash Firmware
Builds (if needed) and uploads the full application binary to the `app0` partition.

In [ ]:
run_pio(["-t", "upload"])

## Flash SPIFFS Partition
Builds the SPIFFS image from `data/` and uploads it to the `spiffs` partition (offset `0x290000`, size `1.4 MB`).  
Use this after modifying `index.html` or other web assets.

In [ ]:
run_pio(["-t", "uploadfs"])

## Build Only (no upload)
Compiles the firmware and SPIFFS image without flashing.

In [ ]:
# Build firmware
cmd = ["pio", "run", "-e", PIO_ENV]
print(f"$ {' '.join(cmd)}")
print("=" * 60)
proc = subprocess.Popen(cmd, cwd=str(FIRMWARE_DIR), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print("=" * 60)
print("Done." if proc.returncode == 0 else f"FAILED (exit {proc.returncode})")

## Erase Flash
Performs a full chip erase. **All data including NVS, firmware, and SPIFFS will be lost.**

In [ ]:
if UPLOAD_PORT is None:
    print("ERROR: No port selected.")
else:
    confirm = input(f"Erase entire flash on {UPLOAD_PORT}? Type 'yes' to confirm: ")
    if confirm.strip().lower() == "yes":
        cmd = ["pio", "run", "-e", PIO_ENV, "-t", "erase", "--upload-port", UPLOAD_PORT]
        print(f"$ {' '.join(cmd)}")
        print("=" * 60)
        proc = subprocess.Popen(cmd, cwd=str(FIRMWARE_DIR), stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True)
        proc.wait()
        print("=" * 60)
        print("Erase complete." if proc.returncode == 0 else f"FAILED (exit {proc.returncode})")
    else:
        print("Aborted.")

## Serial Monitor
Opens the serial monitor on `MONITOR_PORT` at 115200 baud.  
**Interrupt the kernel** (■ button) to stop.

In [ ]:
if not SERIAL_AVAILABLE:
    print("pyserial not available.")
elif MONITOR_PORT is None:
    print("ERROR: No monitor port selected.")
else:
    print(f"Opening serial monitor on {MONITOR_PORT} @ {BAUD} baud")
    print("(Interrupt kernel to stop)")
    print("=" * 60)
    try:
        with serial.Serial(MONITOR_PORT, BAUD, timeout=1) as ser:
            while True:
                line = ser.readline()
                if line:
                    print(line.decode("utf-8", errors="replace"), end="", flush=True)
    except KeyboardInterrupt:
        print("\n[Monitor stopped]")
    except serial.SerialException as e:
        print(f"Serial error: {e}")

## Partition Info
Reads and displays the actual partition table from the connected device.

In [ ]:
if UPLOAD_PORT is None:
    print("ERROR: No port selected.")
else:
    # Use esptool via pio to read partition table
    cmd = [
        "pio", "pkg", "exec", "-p", "tool-esptoolpy", "--",
        "esptool.py", "--port", UPLOAD_PORT,
        "--baud", "921600",
        "read_flash", "0x8000", "0xC00", "partition_table_dump.bin"
    ]
    print(f"Reading partition table from {UPLOAD_PORT}...")
    proc = subprocess.run(cmd, cwd=str(FIRMWARE_DIR), capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode == 0:
        # Decode with gen_esp32part.py
        decode_cmd = [
            "pio", "pkg", "exec", "-p", "tool-esptoolpy", "--",
            "python", "-c",
            "import esp_idf_size.gen_esp32part as g; t=g.PartitionTable.from_binary(open('partition_table_dump.bin','rb').read()); print(t.to_csv())"
        ]
        r2 = subprocess.run(decode_cmd, cwd=str(FIRMWARE_DIR), capture_output=True, text=True)
        if r2.returncode == 0:
            print(r2.stdout)
        else:
            print("(Could not decode — raw binary saved to partition_table_dump.bin)")
    else:
        print(proc.stderr)